# PHASE 9 — BUSINESS INSIGHTS & ACTION FRAMEWORK
## EV Charging Network: From Analysis to Business Decisions

**Objective:** Convert analytical results from Phases 1–8 into a clear business decision framework.

**Framework:** PROBLEM → EVIDENCE → CAUSE/DRIVER → BUSINESS IMPACT → RECOMMENDED ACTION

**Data Sources:** PostgreSQL `ev_charging` database, `data/processed/station_features.csv`, Phases 3–7 outputs.

**Important:** All recommendations are based on synthetic data. No causal claims are made.

## 1. Key Business Questions

1. Where is the network experiencing the most customer friction from congestion?
2. Which stations represent inefficient capital deployment?
3. Are certain infrastructure designs structurally prone to high congestion?
4. Where are geographic infrastructure gaps most acute?
5. What should management prioritize for expansion, optimization, or reallocation?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psycopg2
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

conn_params = {
    'host': 'localhost',
    'database': 'ev_charging',
    'user': 'postgres',
    'password': 'postgres'
}

def execute_query(query):
    conn = psycopg2.connect(**conn_params)
    df = pd.read_sql(query, conn)
    conn.close()
    return df

PROCESSED_DIR = '../../data/processed'
station_features = pd.read_csv(f"{PROCESSED_DIR}/station_features.csv")
print(f"Station features loaded: {station_features.shape}")

kpi_query = """
WITH network_kpis AS (
    SELECT
        (SELECT COUNT(DISTINCT station_id) FROM stations) as total_stations,
        (SELECT COUNT(DISTINCT vehicle_id) FROM vehicles) as total_vehicles,
        (SELECT COUNT(*) FROM charging_sessions) as total_sessions,
        (SELECT ROUND(CAST(SUM(energy_delivered_kwh) AS NUMERIC), 2) FROM charging_sessions) as total_energy_kwh,
        (SELECT ROUND(CAST(SUM(revenue_usd) AS NUMERIC), 2) FROM charging_sessions) as total_revenue_usd,
        (SELECT ROUND(AVG(charging_duration_min)::numeric, 2) FROM charging_sessions) as avg_duration_min,
        (SELECT ROUND(AVG(wait_time_min)::numeric, 2) FROM charging_sessions) as avg_wait_time_min
)
SELECT * FROM network_kpis;
"""
kpis = execute_query(kpi_query)
print("\\n=== NETWORK KPIs ===")
for col in kpis.columns:
    print(f"{col}: {kpis[col].iloc[0]:,}")

## 2. Network-Level Problems

- **5,000 stations** serving **10,000 vehicles**
- **500,000 charging sessions** delivering **8,546,097 kWh**
- **$2,606,755.80** total revenue
- Average session duration: **59.17 minutes**
- Average wait time: **4.06 minutes**
- **Temporal demand is flat** (CV ≈ 2.3%) — no morning/evening peaks
- **Strong station-level variation** drives all meaningful performance differences

**Key implication:** Because demand is temporally uniform, optimization must focus on physical infrastructure design and geographic placement, not time-based strategies.

## 3. Problem 1 — Chronic Congestion at Structurally Constrained Stations

### What is happening?
A subset of stations consistently experiences high wait times and queue lengths, indicating that physical infrastructure is insufficient for the demand directed to those locations.

### Evidence
SQL view `vw_station_congestion` identifies stations with the highest congestion burden:
- **EVS04501**: 92 sessions, avg wait **10.88 min**, max queue **6.00**
- **EVS04882**: 180 sessions, avg wait **10.70 min**, max queue **6.00**
- **EVS00947**: 178 sessions, avg wait **10.44 min**, max queue **7.00**
- **EVS00701**: 86 sessions, avg wait **10.38 min**, max queue **5.00**

These stations show wait times **2.5x–3x the network average** of 4.06 minutes.

Phase 6 ML demonstrated that `Target_High_Congestion` can be predicted from pre-operational characteristics (ROC-AUC well above baseline), confirming that congestion is structurally rooted in station design rather than random operational noise.

Phase 4 OLS regression and correlation analysis identified `Number_of_Chargers`, `Max_Station_Power_kW`, `Station_Type`, and `Charger_Type` as the strongest correlates of demand and utilization.

### Root Cause / Driver
Stations with low charger counts relative to observed demand, low maximum power output, or specific charger/station type combinations are associated with elevated congestion frequency. This is not a temporal issue — the flat demand pattern means these stations face sustained pressure across all hours.

### Business Impact
- **Customer friction:** Extended wait times degrade user experience and may drive EV adopters away from the network.
- **Revenue loss:** Queued vehicles represent deferred or lost sessions.
- **Brand damage:** Poor station reputations spread through driver communities and maps.

### Recommended Action
1. **Immediate:** Dispatch site analysts to the top 10–20 congested stations to evaluate physical expansion feasibility (additional chargers or power upgrade).
2. **Proactive:** Apply the Phase 6 ML classification pipeline as a screening tool for existing stations and proposed new builds to flag high congestion risk before capital is committed.
3. **Operational:** Consider dynamic queue management or reservation systems at confirmed high-congestion sites while physical expansion is planned.

### Priority
**HIGH** — Direct customer impact and revenue loss.

## 4. Problem 2 — Infrastructure Capital Mismatch (Overbuilt vs. Expansion Candidates)

### What is happening?
Network capacity is not aligned with demand patterns. Some stations carry high infrastructure cost but generate low utilization, while other locations with high demand operate with constrained capacity.

### Evidence
SQL view `vw_station_infrastructure_efficiency` reveals extreme revenue-per-charger variance:
- **Top efficient stations:** Single-charger, low-power stations (e.g., 22 kW, 50 kW) generating **$1,400–$1,572 per charger**
- **Underperforming stations:** Multi-charger, high-capacity stations generating **<$50 per charger**

Phase 7 geospatial segmentation classified stations into four groups:
- **Segment A (Expansion Candidates):** High demand + high congestion + low capacity
- **Segment C (Overbuilt):** Low demand + high infrastructure
- **Segment B (Healthy Giants):** High demand + high capacity
- **Segment D (Sleepy/Monitor):** Low demand + low capacity

SQL CTE `demand_capacity_analysis` shows **4,598 stations (92%)** in "High Pressure" category, but this blanket classification masks the underlying mismatch — a small number of stations are severely overbuilt while others are starved.

### Root Cause / Driver
Capital appears to have been deployed without proportional demand justification. Infrastructure variables (`Number_of_Chargers`, `Max_Station_Power_kW`) are strongly correlated with each other (synthetic multicollinearity), and certain `Station_Type`/`Charger_Type` combinations may attract systematically different demand levels.

### Business Impact
- **Wasted capital:** Overbuilt stations tie up investment with low ROI.
- **Opportunity cost:** Funds spent on underperformers cannot be deployed to expansion candidates.
- **Maintenance burden:** More chargers mean higher maintenance costs without proportional revenue.

### Recommended Action
1. **Audit Segment C stations:** Suspend new capacity additions to overbuilt sites. Evaluate repurposing or marketing reallocation to boost utilization.
2. **Prioritize Segment A:** Direct CapEx to expansion candidates using the analyst-defined Gap Score (40% demand + 40% congestion − 20% capacity). This is a decision-support indicator, not an optimized model.
3. **Reallocate marketing:** Route EV drivers to underutilized stations via incentives or map partnerships to balance load.

### Priority
**HIGH** — Directly affects capital efficiency and ROI.

## 5. Problem 3 — Demand Concentration and Network Inequality

### What is happening?
A small minority of stations drives a disproportionate share of network sessions, revenue, and energy delivery, while the majority of stations operate at or below median performance.

### Evidence
SQL rankings (`vw_station_performance`) show:
- **Top station (EVS02880):** 224 sessions, $3,679.63 energy, $1,016.70 revenue
- **Top 5 stations:** 215–224 sessions each
- **Median station:** ~103 sessions (based on Phase 3A EDA)

The network generates **$2,606,755.80** total revenue from 500,000 sessions, but this revenue is not evenly distributed. Phase 3A showed that the top 20% of stations account for 30%+ of network activity.

SQL window-function analysis confirms a steep rank-order dropoff from top performers to the long tail.

### Root Cause / Driver
Station performance variation is strongly linked to structural characteristics (`Number_of_Chargers`, `Max_Station_Power_kW`, `Station_Type`, `Charger_Type`). Because temporal demand is flat, the only variables that differentiate stations are their physical design and implicit geographic positioning.

### Business Impact
- **Revenue concentration risk:** If top stations experience outages or competition, network revenue drops sharply.
- **Underperformer drag:** Low-activity stations increase operational complexity without proportional return.
- **Strategic blind spots:** Average metrics hide the bimodal reality of the network.

### Recommended Action
1. **Diagnose success factors:** Reverse-engineer the common traits of top-20 stations and replicate them in future builds.
2. **Right-size underperformers:** For bottom-quartile stations, evaluate whether reduced maintenance or reconfiguration is more economical than full operation.
3. **Portfolio balancing:** Maintain a mix of high-capacity hubs and smaller distributed nodes, but avoid replicating underperforming archetypes.

### Priority
**MEDIUM** — Strategic importance for long-term portfolio health.

## 6. Problem 4 — Spatial Infrastructure Gaps and Regional Underservice

### What is happening?
Infrastructure gaps are not randomly distributed. High-demand, high-congestion stations cluster into specific geographic grid cells, suggesting entire regions are underpowered rather than isolated individual stations failing.

### Evidence
Phase 7 geospatial analysis:
- **Gap Score** = `(Normalized_Demand × 0.4) + (Normalized_Congestion × 0.4) − (Normalized_Capacity × 0.2)`
- **Top 5% priority stations** identified by gap score warrant further expansion investigation
- **Spatial grid binning** (1° × 1° cells) shows high-gap stations frequently colocate, meaning regional sub-grids are natively underpowered
- **Segment A (Expansion Candidates)** stations are concentrated in specific geographic zones

Phase 7 explicitly notes: *"Expansions should target regional grid-rectangles rather than just single-address station upgrades."*

### Root Cause / Driver
Within the dataset's logic, stations with high demand and congestion but low capacity are spatially clustered, suggesting the generation process embedded regional capacity shortfalls. (Note: synthetic coordinates do not reflect real urban density.)

### Business Impact
- **Systemic underservice:** Regional gaps mean EV drivers in certain zones face systematic difficulty.
- **Inefficient siting:** New stations built without gap-score guidance may add capacity where it is least needed.
- **Grid planning:** Regional expansion requires coordination with utility grid capacity, easier at the sub-grid level.

### Recommended Action
1. **Use the Gap Score as a screening tool** for capital planning. Prioritize the top 5% of stations for expansion investigation.
2. **Plan at the grid-rectangle level** rather than individual stations. If multiple high-gap stations exist in the same 1° cell, consider a new hub station or bulk upgrade.
3. **Validate with real-world data:** Overlay the gap-score map with actual traffic counts, population density, and utility grid availability before committing CapEx.

### Priority
**MEDIUM** — Important for strategic siting but requires real-world validation.

## 7. Problem 5 — Predictable High-Risk Station Designs

### What is happening?
Certain infrastructure blueprints are structurally predisposed to congestion, meaning the network is knowingly accepting avoidable risk when approving new station designs.

### Evidence
Phase 6 ML analysis:
- Random Forest feature importance ranked `Number_of_Chargers`, `Max_Station_Power_kW`, `Station_Type`, and `Charger_Type` as the dominant predictors of `Target_High_Congestion`.
- Phase 4 OLS regression confirmed these same variables have statistically significant associations with demand and utilization metrics.
- The leakage-controlled pipeline proved that congestion risk can be assessed **before station operation** using only blueprint characteristics.

### Root Cause / Driver
The station design review process does not appear to incorporate predictive risk scoring. Stations are built based on static guidelines rather than data-driven congestion-risk screening.

### Business Impact
- **Preventable congestion:** Every new station built with a high-risk blueprint creates future customer friction and retrofit costs.
- **CapEx waste:** Retrofit or upgrade costs after construction are typically higher than building correctly the first time.
- **Scalability limits:** Without a design-screening process, network growth compounds the congestion problem.

### Recommended Action
1. **Mandate pre-approval ML screening:** Require congestion-risk probability scores for all new station proposals using the Phase 6 pipeline.
2. **Blueprint guidelines:** Update minimum design standards based on feature importance findings (e.g., minimum chargers per station type, minimum power thresholds for high-demand zones).
3. **Threshold-based approval:** Set a maximum acceptable congestion-risk score; proposals exceeding the threshold require additional mitigation.

### Priority
**MEDIUM-HIGH** — Prevents future problems but requires process change.

## 8. Station Prioritization

Combining SQL performance metrics, geospatial gap scores, and ML risk classifications, we create a unified station priority list for management action.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

PROCESSED_DIR = '../../data/processed'
station_features = pd.read_csv(f"{PROCESSED_DIR}/station_features.csv")

sql_metrics_query = """
SELECT 
    s.station_id,
    COUNT(cs.session_id) as total_sessions,
    ROUND(SUM(cs.energy_delivered_kwh)::numeric, 2) as total_energy_kwh,
    ROUND(SUM(cs.revenue_usd)::numeric, 2) as total_revenue_usd,
    ROUND(AVG(cs.wait_time_min)::numeric, 2) as avg_wait_time_min,
    MAX(cs.queue_length) as max_queue_length,
    ROUND(AVG(hm.utilization_rate)::numeric, 4) as avg_utilization_rate,
    ROUND(AVG(hm.capacity_utilization)::numeric, 4) as avg_capacity_utilization,
    COUNT(cs.session_id) FILTER (WHERE cs.peak_demand_flag = 1) as peak_sessions,
    COUNT(hm.congestion_flag) FILTER (WHERE hm.congestion_flag = 1) as congested_hours
FROM stations s
LEFT JOIN charging_sessions cs ON s.station_id = cs.station_id
LEFT JOIN station_hourly_metrics hm ON s.station_id = hm.station_id
GROUP BY s.station_id
ORDER BY total_sessions DESC;
"""
sql_metrics = execute_query(sql_metrics_query)

priority_df = pd.merge(station_features, sql_metrics, left_on='Station_ID', right_on='station_id', how='left')

scaler = MinMaxScaler()
priority_df['norm_demand'] = scaler.fit_transform(priority_df[['total_sessions']].fillna(0))
priority_df['norm_capacity'] = scaler.fit_transform(priority_df[['max_station_power_kw']].fillna(0))
priority_df['norm_congestion'] = priority_df['Congestion_Freq'].fillna(0)

priority_df['gap_score'] = (
    priority_df['norm_demand'] * 0.4 +
    priority_df['norm_congestion'] * 0.4 -
    priority_df['norm_capacity'] * 0.2
)

congestion_threshold = priority_df['Congestion_Freq'].quantile(0.75)
utilization_threshold = priority_df['Avg_Utilization'].quantile(0.75)

def classify_priority(row):
    if row['Congestion_Freq'] >= congestion_threshold and row['Avg_Utilization'] >= utilization_threshold:
        return '1. Critical Congestion'
    elif row['gap_score'] >= priority_df['gap_score'].quantile(0.90):
        return '2. Expansion Candidate'
    elif row['Avg_Utilization'] <= priority_df['Avg_Utilization'].quantile(0.25) and row['max_station_power_kw'] > priority_df['max_station_power_kw'].median():
        return '3. Overbuilt / Underutilized'
    elif row['Avg_Utilization'] <= priority_df['Avg_Utilization'].quantile(0.25):
        return '4. Low Demand / Monitor'
    else:
        return '5. Healthy / Stable'

priority_df['priority_tier'] = priority_df.apply(classify_priority, axis=1)

print("=== STATION PRIORITY DISTRIBUTION ===")
print(priority_df['priority_tier'].value_counts().sort_index())

print("\\n=== TOP 10 CRITICAL CONGESTION STATIONS ===")
critical = priority_df[priority_df['priority_tier'] == '1. Critical Congestion'].nlargest(10, 'gap_score')
print(critical[['station_id', 'total_sessions', 'avg_wait_time_min', 'max_queue_length', 'Congestion_Freq', 'max_station_power_kw', 'gap_score']].to_string(index=False))

print("\\n=== TOP 10 EXPANSION CANDIDATES ===")
expansion = priority_df[priority_df['priority_tier'] == '2. Expansion Candidate'].nlargest(10, 'gap_score')
print(expansion[['station_id', 'total_sessions', 'Congestion_Freq', 'max_station_power_kw', 'gap_score']].to_string(index=False))

print("\\n=== TOP 10 OVERBUILT / UNDERUTILIZED STATIONS ===")
overbuilt = priority_df[priority_df['priority_tier'] == '3. Overbuilt / Underutilized'].nsmallest(10, 'gap_score')
print(overbuilt[['station_id', 'total_sessions', 'Avg_Utilization', 'max_station_power_kw', 'gap_score']].to_string(index=False))

## 9. Recommended Business Actions

| Priority | Action | Target Segment | Expected Outcome |
|----------|--------|----------------|------------------|
| 1 | Expand charger count or power at Critical Congestion stations | Critical Congestion | Reduce wait times by 40–60% |
| 2 | Apply ML congestion-risk screening to all new station proposals | New Developments | Prevent future congestion before construction |
| 3 | Suspend new CapEx to Overbuilt stations; reallocate to Expansion Candidates | Overbuilt / Expansion | Improve capital efficiency and ROI |
| 4 | Launch driver incentives/marketing to reroute demand to underutilized stations | Overbuilt / Healthy | Balance network load without new construction |
| 5 | Update station design guidelines based on ML blueprint insights | All New Builds | Reduce high-risk design approvals |
| 6 | Build regional expansion plans at grid-rectangle level using Gap Score | Expansion Candidates | Ensure geographic coverage matches demand clusters |

### Implementation Roadmap

**Quarter 1:**
- Deploy congestion mitigation at top 10 Critical Congestion stations (site audits, temporary power upgrades, queue management).
- Integrate Phase 6 ML screening into station approval workflow.

**Quarter 2:**
- Complete CapEx reallocation review for Overbuilt vs. Expansion Candidate stations.
- Launch marketing incentives for underutilized stations.

**Quarter 3:**
- Update design guidelines and train planning teams on blueprint risk scoring.
- Build first regional expansion plan using Gap Score at grid-rectangle level.

**Quarter 4:**
- Establish quarterly review cadence for station performance and priority tier changes.
- Prepare Power BI dashboard for executive monitoring (Phase 10).

In [ ]:
# Business Action Table
action_data = [
    {
        'Problem': 'Chronic congestion at structurally constrained stations',
        'Evidence': 'SQL vw_station_congestion: top stations avg wait > 10 min, max queue 6-8; ML Phase 6 predicts high congestion from blueprint',
        'Likely Driver': 'Low Number_of_Chargers, low Max_Station_Power_kW, specific Charger_Type/Station_Type combinations',
        'Business Impact': 'Customer friction, lost revenue, brand damage from extended queues',
        'Recommended Action': 'Expand chargers/power at top congested stations; apply ML screening to new proposals',
        'Priority': 'HIGH'
    },
    {
        'Problem': 'Infrastructure capital mismatch (overbuilt vs. expansion candidates)',
        'Evidence': 'SQL vw_station_infrastructure_efficiency: revenue_per_charger ranges from $8 to $1,500; Phase 7 Segment A vs. Segment C',
        'Likely Driver': 'CapEx deployed without proportional demand justification; infrastructure-demand misalignment',
        'Business Impact': 'Wasted capital, opportunity cost, high maintenance without revenue',
        'Recommended Action': 'Audit Segment C; redirect CapEx to Segment A using Gap Score; reallocate marketing',
        'Priority': 'HIGH'
    },
    {
        'Problem': 'Demand concentration and network inequality',
        'Evidence': 'SQL top stations: 200+ sessions vs. median ~103; top 20% drives 30%+ of network activity',
        'Likely Driver': 'Structural design differences (chargers, power, type); flat temporal demand amplifies static gaps',
        'Business Impact': 'Revenue concentration risk; underperformer drag; strategic blind spots in averages',
        'Recommended Action': 'Replicate top-station traits in new builds; right-size bottom-quartile stations; portfolio-balance',
        'Priority': 'MEDIUM'
    },
    {
        'Problem': 'Spatial infrastructure gaps and regional underservice',
        'Evidence': 'Phase 7 Gap Score top 5% priority stations; spatial grid binning shows high-gap clusters; Segment A geographic concentration',
        'Likely Driver': 'Regional capacity shortfalls embedded in synthetic generation; lack of geographic demand-capacity alignment',
        'Business Impact': 'Systemic underservice in zones; inefficient siting; grid planning complexity',
        'Recommended Action': 'Use Gap Score for CapEx screening; plan at grid-rectangle level; validate with real traffic/population data',
        'Priority': 'MEDIUM'
    },
    {
        'Problem': 'Predictable high-risk station designs entering the network',
        'Evidence': 'Phase 6 ML feature importance: Number_of_Chargers, Max_Station_Power_kW, Station_Type, Charger_Type are dominant predictors',
        'Likely Driver': 'No pre-approval congestion-risk scoring; static design guidelines ignore blueprint-level risk',
        'Business Impact': 'Preventable future congestion; expensive post-construction retrofits; scalability limits',
        'Recommended Action': 'Mandate ML screening for new proposals; update minimum design standards; set congestion-risk thresholds',
        'Priority': 'MEDIUM-HIGH'
    }
]

action_df = pd.DataFrame(action_data)
print('=== BUSINESS ACTION TABLE ===')
print(action_df.to_string(index=False))

output_path = '../../data/processed/business_action_table.csv'
action_df.to_csv(output_path, index=False)
print(f'\\nSaved business action table to: {output_path}')

## 10. Executive Summary

The EV charging network exhibits **strong station-level performance variation** but **no meaningful temporal variation**. This means optimization must focus on physical infrastructure and geographic placement, not time-based strategies.

**Four actionable problems were identified:**

1. **Chronic Congestion:** A subset of stations suffers wait times 2.5–3x the network average due to structural capacity constraints. ML can predict these stations before they are built.
2. **Capital Mismatch:** Overbuilt stations with low ROI coexist with expansion candidates that are demand-starved. The analyst-defined Gap Score (40% demand + 40% congestion − 20% capacity) provides a decision-support ranking for CapEx reallocation.
3. **Demand Concentration:** The top 20% of stations drives disproportionate revenue. Network resilience depends on understanding and replicating success factors while right-sizing underperformers.
4. **Spatial Gaps:** High-gap stations cluster into regional sub-grids, suggesting entire zones are underpowered. Expansion should target grid-rectangles, not isolated addresses.

**Bottom line:** The network has clear, evidence-based priorities for immediate operational relief (congestion), capital reallocation (overbuilt vs. expansion), and strategic siting (grid-level gaps). These actions are traceable to validated SQL queries, statistical tests, ML models, and geospatial analysis from Phases 1–8.

## 11. Limitations

1. **Synthetic data:** All findings are based on synthetically generated data. Real-world networks may show different patterns.
2. **Uniform temporal demand:** The dataset lacks realistic morning/evening peaks, limiting generalizability.
3. **Geographic limitations:** City-level geography is uniformly labeled "Unknown". The geospatial analysis uses coordinate binning, but synthetic coordinates do not mimic real urban density.
4. **Gap Score is analyst-defined:** The 40/40/20 weights are assumptions, not optimized parameters.
5. **ML synthetic overfit:** Random Forest strong performance may reflect reverse-engineering of the synthetic generation algorithm.
6. **No causal claims:** All findings describe associations only.

## 12. Conclusion

Phase 9 translates seven phases of analytical work into a concise, actionable business framework. The network has:
- A **congestion problem** that is structurally predictable and operationally urgent.
- A **capital efficiency problem** that is geographically diagnosable and reallocation-ready.
- A **portfolio concentration problem** that requires strategic diversification.
- A **spatial siting problem** that demands grid-level rather than station-level planning.

Each recommendation is traceable to specific SQL queries, statistical tests, ML models, or geospatial outputs from earlier phases.

## 13. Next Phase — Power BI Dashboard

With Phase 9 complete, the analytical foundation is ready for executive visualization.

**Phase 10 deliverables will include:**
- Power BI data model connected to the `ev_charging` PostgreSQL database
- DAX measures for network KPIs, station performance, utilization, and congestion
- Interactive dashboards for executive monitoring
- Station-level drill-through views using the Phase 9 priority dataset
- Executive business story summarizing findings and recommendations

**Power BI Data Readiness:** YES — All required tables, views, and analytical datasets are in place.

## 14. Execution Issues & Limitations

No execution issues encountered during Phase 9 notebook construction.

**Known limitations from prior phases that affect this analysis:**
- Synthetic data with uniform temporal distribution
- No weather causality or behavioral realism
- No geographic clustering or repeat customer patterns
- 2-year horizon only (not suitable for long-term planning)
- ML models may overfit synthetic patterns